In [4]:
import os
import librosa
import numpy as np

# הנתיב המדויק לתיקיית הרגשות ב-Kaggle
dataset_path = "C:\\Users\\amitn\\OneDrive\\שולחן העבודה\\kaggle\\Toronto emotional speech set (TESS)\\TESS Toronto emotional speech set data"

X = []  # כאן נשמרים הפיצ'רים
y = []  # כאן נשמרים התיוגים (רגשות)

# לולאה על כל תיקיות הרגשות
for folder in os.listdir(dataset_path):
    folder_path = os.path.join(dataset_path, folder)
    if os.path.isdir(folder_path):
        emotion_label = folder.split('_')[-1]  # מחלץ את שם הרגש מהתיקיה
        
        # לולאה על כל קובץ WAV בתיקיה
        for file in os.listdir(folder_path):
            if file.endswith('.wav'):
                file_path = os.path.join(folder_path, file)
                
                # 1️⃣ טעינת הקובץ
                signal, sr = librosa.load(file_path, sr=None)
                
                # 2️⃣ פרי-פרוססינג
                # הסרת שקט בתחילת ובסוף הקובץ
                signal, _ = librosa.effects.trim(signal)
                
                # נירמול האות [-1, 1]
                signal = signal / np.max(np.abs(signal))
                
                # 3️⃣ חילוץ MFCC (וקטור קבוע לכל קובץ)
                mfcc = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=30)
                features = np.mean(mfcc.T, axis=0)
                
                # שמירת הפיצ'רים והתווית
                X.append(features)
                y.append(emotion_label)

# המרה למערכים numpy
X = np.array(X)
y = np.array(y)

print("Data loaded. Samples:", X.shape[0])
print("Feature shape per sample:", X.shape[1])

Data loaded. Samples: 2800
Feature shape per sample: 30


In [5]:
import pandas as pd

# נניח שיש לנו X (MFCCs) ו-y (רגשות)
# יוצרים DataFrame
mfcc_columns = [f"MFCC{i+1}" for i in range(X.shape[1])]
df_features = pd.DataFrame(X, columns=mfcc_columns)
df_features['label'] = y  # מוסיפים את הרגש

# הצגה של 5 השורות הראשונות
print(df_features.head())

        MFCC1      MFCC2      MFCC3      MFCC4      MFCC5      MFCC6  \
0 -283.245728  55.235966 -15.034652 -10.704583  -6.318043   9.424741   
1 -280.363708  67.048828  -0.448259 -16.812130 -14.137628  12.182583   
2 -276.064209  28.609005  -5.317216   2.124631  -3.238452   5.009407   
3 -281.394989  53.439651  -8.663544 -13.309977  -5.566482   8.417087   
4 -280.501251  54.462574   3.355292  10.967580   2.167008   6.005212   

       MFCC7      MFCC8      MFCC9     MFCC10  ...    MFCC22    MFCC23  \
0 -20.741861 -12.001626  -7.151770  -7.919953  ... -1.346145 -7.081009   
1  -8.768966  -0.833090 -12.662774   1.033059  ... -0.595418 -4.815976   
2 -20.190367  -2.519339 -12.905814  -1.877586  ...  3.915622  0.272890   
3 -18.068228 -11.134799 -10.535918  -8.044549  ... -0.024202 -8.031614   
4 -25.438648   4.935780 -12.994332 -16.313234  ...  2.791510  1.926306   

      MFCC24     MFCC25     MFCC26     MFCC27     MFCC28     MFCC29  \
0  -0.296030  11.390141  14.699961  25.256971  24.0

In [6]:
import pandas as pd

# נניח X הוא מערך numpy בגודל (num_samples, num_features)
# y = רשימת התוויות

# שמות כל העמודות (MFCC1–MFCC13 לדוגמה)
all_columns = [
    'MFCC1', 'MFCC2', 'MFCC3', 'MFCC4', 'MFCC5',
    'MFCC6', 'MFCC7', 'MFCC8', 'MFCC9', 'MFCC10',
    'MFCC11', 'MFCC12', 'MFCC13', 'MFCC14', 'MFCC15',
    'MFCC16', 'MFCC17', 'MFCC18', 'MFCC19', 'MFCC20',
    'MFCC21', 'MFCC22', 'MFCC23', 'MFCC24', 'MFCC25',
    'MFCC26', 'MFCC27', 'MFCC28', 'MFCC29', 'MFCC30'
]



# המרה ל-DataFrame
df_features = pd.DataFrame(X, columns=all_columns)

# רשימת הפיצ'רים למחיקה
features_to_drop = [
                    
               ]

# מחיקה ללא שינוי המקורי
df_new = df_features.drop(columns=features_to_drop)

# עכשיו df_new מוכן לאימון
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

# קידוד התוויות
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# פיצול Train/Test
X_train, X_test, y_train, y_test = train_test_split(
    df_new, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# נירמול
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# אימון Random Forest
clf = RandomForestClassifier(n_estimators=200, random_state=42)
clf.fit(X_train_scaled, y_train)

# תחזיות
y_train_pred = clf.predict(X_train_scaled)
y_test_pred = clf.predict(X_test_scaled)

# Accuracy ו-F1 לכל סט
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)
train_f1 = f1_score(y_train, y_train_pred, average='weighted')
test_f1 = f1_score(y_test, y_test_pred, average='weighted')

print(f"Train Accuracy: {train_accuracy:.4f}")
print(f"Train F1-score: {train_f1:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test F1-score: {test_f1:.4f}")

print("\nClassification Report (Test Set):\n")
print(classification_report(y_test, y_test_pred, target_names=le.classes_))


Train Accuracy: 1.0000
Train F1-score: 1.0000
Test Accuracy: 0.9946
Test F1-score: 0.9946

Classification Report (Test Set):

              precision    recall  f1-score   support

        Fear       0.98      1.00      0.99        40
         Sad       1.00      1.00      1.00        40
       angry       0.99      0.99      0.99        80
     disgust       1.00      1.00      1.00        80
        fear       1.00      1.00      1.00        40
       happy       0.99      1.00      0.99        80
     neutral       1.00      1.00      1.00        80
         sad       1.00      1.00      1.00        40
    surprise       1.00      0.97      0.99        40
   surprised       1.00      0.97      0.99        40

    accuracy                           0.99       560
   macro avg       1.00      0.99      0.99       560
weighted avg       0.99      0.99      0.99       560



In [10]:
import sounddevice as sd
from scipy.io.wavfile import write

# פרמטרים
duration = 2  # משך ההקלטה (שניות)
fs = 44100    # תדירות דגימה (Hz)
filename = "recording.wav"

print("🎤 מתחיל הקלטה...")
recording = sd.rec(int(duration * fs), samplerate=fs, channels=1, dtype='int16')
sd.wait()  # ממתין לסיום ההקלטה
print("✅ סיום ההקלטה.")

# שמירת הקובץ בפורמט WAV
write(filename, fs, recording)

print(f"קובץ נשמר בשם: {filename}")


🎤 מתחיל הקלטה...
✅ סיום ההקלטה.
קובץ נשמר בשם: recording.wav


In [23]:
joblib.dump(le,"le.pkl")


['le.pkl']

In [29]:
joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']

In [11]:
import joblib

# נניח שהמודל שלך מאומן בשם clf
joblib.dump(clf, "model.pkl")

['model.pkl']

In [64]:
import sounddevice as sd
import numpy as np
import librosa
import joblib

DURATION = 2  # שניות
SAMPLE_RATE = 24414  # קצב הדגימה של המיקרופון והמודל

# טוענים את המודל ואת הקידוד
model = joblib.load(r"C:\Users\amitn\OneDrive\שולחן העבודה\kaggle\Toronto emotional speech set (TESS)\TESS Toronto emotional speech set data\model.pkl")
le = joblib.load(r"C:\Users\amitn\OneDrive\שולחן העבודה\kaggle\Toronto emotional speech set (TESS)\TESS Toronto emotional speech set data\le.pkl")

def record_and_predict():
    print("🎤 מתחיל הקלטה...")
    recording = sd.rec(int(DURATION * SAMPLE_RATE), samplerate=SAMPLE_RATE, channels=1, dtype='float32')
    sd.wait()
    print("✅ הקלטה הסתיימה.")

    audio = recording.flatten()

    # הסרת שקט בתחילת ובסוף הקובץ
    audio_trimmed, _ = librosa.effects.trim(audio)

    # נירמול [-1, 1]
    audio_normalized = audio_trimmed / np.max(np.abs(audio_trimmed))

    # חילוץ MFCC
    mfcc = librosa.feature.mfcc(y=audio_normalized, sr=SAMPLE_RATE, n_mfcc=30)
    features = np.mean(mfcc.T, axis=0)

    # חיזוי
    pred_encoded = model.predict([features])
    pred_label = le.inverse_transform(pred_encoded)
    print(f"החיזוי: {pred_label[0]}")
    return pred_label[0]

# דוגמה לשימוש
if __name__ == "__main__":
    result = record_and_predict()


🎤 מתחיל הקלטה...
✅ הקלטה הסתיימה.
החיזוי: angry
